In [1]:
# ============================================
# RETAIL CUSTOMER SEGMENTATION PROJECT
# ============================================

# ============================================
# STEP 1: IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

# ============================================
# STEP 2: LOAD DATASET
# ============================================

# Replace with your dataset path
df = pd.read_csv("Mall_Customers.csv")

# Display first 5 rows
print(df.head())

# Dataset info
print(df.info())

# Check missing values
print(df.isnull().sum())

# ============================================
# STEP 3: DATA PREPROCESSING
# ============================================

# Rename columns (optional)
df.rename(columns={
    'Annual Income (k$)': 'Annual_Income',
    'Spending Score (1-100)': 'Spending_Score'
}, inplace=True)

# Encode Gender column
df['Gender'] = df['Gender'].map({
    'Male': 0,
    'Female': 1
})

# ============================================
# STEP 4: EXPLORATORY DATA ANALYSIS
# ============================================

# Age Distribution
plt.figure(figsize=(8,5))
sns.histplot(df['Age'], bins=20, kde=True)
plt.title("Age Distribution")
plt.show()

# Annual Income Distribution
plt.figure(figsize=(8,5))
sns.histplot(df['Annual_Income'], bins=20, kde=True)
plt.title("Annual Income Distribution")
plt.show()

# Spending Score Distribution
plt.figure(figsize=(8,5))
sns.histplot(df['Spending_Score'], bins=20, kde=True)
plt.title("Spending Score Distribution")
plt.show()
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(8,6))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Heatmap")

plt.show()
# ============================================
# STEP 5: FEATURE SELECTION
# ============================================

X = df[['Age', 'Annual_Income', 'Spending_Score']]

# ============================================
# STEP 6: FEATURE SCALING
# ============================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================
# STEP 7: ELBOW METHOD
# ============================================

wcss = []

for i in range(1,11):
    kmeans = KMeans(
        n_clusters=i,
        init='k-means++',
        random_state=42
    )
    
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Plot Elbow Curve
plt.figure(figsize=(8,5))
plt.plot(range(1,11), wcss, marker='o')
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.show()

# ============================================
# STEP 8: TRAIN K-MEANS MODEL
# ============================================

kmeans = KMeans(
    n_clusters=5,
    init='k-means++',
    random_state=42
)

clusters = kmeans.fit_predict(X_scaled)

# Add cluster column
df['Cluster'] = clusters

# ============================================
# STEP 9: SILHOUETTE SCORE
# ============================================

score = silhouette_score(X_scaled, clusters)

print("Silhouette Score:", score)

# ============================================
# STEP 10: PCA FOR VISUALIZATION
# ============================================

pca = PCA(n_components=2)

pca_components = pca.fit_transform(X_scaled)

df['PCA1'] = pca_components[:,0]
df['PCA2'] = pca_components[:,1]

# ============================================
# STEP 11: CLUSTER VISUALIZATION
# ============================================

plt.figure(figsize=(10,7))

sns.scatterplot(
    x='PCA1',
    y='PCA2',
    hue='Cluster',
    palette='Set1',
    data=df,
    s=100
)

plt.title("Customer Segments")
plt.show()

# ============================================
# STEP 12: CLUSTER ANALYSIS
# ============================================
# Cluster Summary (only numeric columns)

cluster_summary = df.groupby('Cluster').mean(numeric_only=True)

print(cluster_summary)

# ============================================
# STEP 13: SEGMENT INTERPRETATION
# ============================================

for i in range(5):
    
    print(f"\nCluster {i}")
    print(df[df['Cluster']==i].describe())

# ============================================
# STEP 14: SAVE RESULTS
# ============================================

df.to_csv("Customer_Segments_Output.csv", index=False)

print("Segmentation completed successfully!")


ModuleNotFoundError: No module named 'pandas'